# CogniStream — Data Collection and Initial Inspection

## Objective
Inspect the AI Developer Productivity dataset before performing
data cleaning, exploratory analysis, or statistical modeling.

## Dataset
- Records: 500
- Features: 9
- Source: ai_dev_productivity.csv

## Workflow
1. Load the original dataset.
2. Validate the expected schema.
3. Inspect data types and descriptive statistics.
4. Examine missing values and duplicate records.
5. Validate numeric ranges.
6. Save a reproducible data-quality report.

## Data integrity
The original CSV must remain unchanged.
Data transformations will be performed in separate modules.

In [1]:
from pathlib import Path
import sys

# Locate the project root by finding src/schema.py.
current_dir = Path.cwd().resolve()

candidates = [current_dir, *current_dir.parents]

PROJECT_ROOT = next(
    (
        folder
        for folder in candidates
        if (folder / "src" / "schema.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot find the CogniStream project root. "
        "Open the correct project folder in VS Code."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\PlayGround\Infotact Data Analytics\CogniStream


In [2]:
import pandas as pd
from IPython.display import display

from src.schema import EXPECTED_COLUMNS
from src.data_collection import (
    load_dataset,
    validate_schema,
    inspect_dataset,
)

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "dataset"
    / "raw"
    / "ai_dev_productivity.csv"
)

print("Imports successful.")
print("Expected columns:", len(EXPECTED_COLUMNS))

Imports successful.
Expected columns: 9


In [3]:
raw_df = load_dataset(RAW_DATA_PATH)

print("Dataset shape:", raw_df.shape)
print("Records:", len(raw_df))
print("Features:", len(raw_df.columns))

display(raw_df.head())

Dataset shape: (500, 9)
Records: 500
Features: 9


,hours_coding,coffee_intake_mg,distractions,sleep_hours,commits,bugs_reported,ai_usage_hours,cognitive_load,task_success
0,5.99,600,1,5.8,2,1,0.71,5.4,1
1,4.72,568,2,6.9,5,3,1.75,4.7,1
2,6.30,560,1,8.9,2,0,2.27,2.2,1
3,8.05,600,7,6.3,9,5,1.40,5.9,0
4,4.53,421,6,6.9,4,0,1.26,6.3,1


In [4]:
schema_table = pd.DataFrame({
    "Column": raw_df.columns,
    "Data_Type": raw_df.dtypes.astype(str).values,
    "Non_Missing": raw_df.notna().sum().values,
    "Missing": raw_df.isna().sum().values,
    "Unique_Values": raw_df.nunique().values,
})

display(schema_table)

,Column,Data_Type,Non_Missing,Missing,Unique_Values
0,hours_coding,float64,500,0,353
1,coffee_intake_mg,int64,500,0,244
2,distractions,int64,500,0,9
3,sleep_hours,float64,500,0,67
4,commits,int64,500,0,14
5,bugs_reported,int64,500,0,6
6,ai_usage_hours,float64,500,0,260
7,cognitive_load,float64,500,0,79
8,task_success,int64,500,0,2


In [5]:
schema_report = validate_schema(raw_df)

print("Schema valid:", schema_report["schema_valid"])
print("Missing columns:", schema_report["missing_columns"])
print("Unexpected columns:", schema_report["unexpected_columns"])

assert schema_report["schema_valid"], (
    "Schema validation failed. Resolve column differences before continuing."
)

print("\nSchema validation passed.")

Schema valid: True
Missing columns: []
Unexpected columns: []

Schema validation passed.


In [6]:
quality_report = inspect_dataset(raw_df)

print("=" * 55)
print("COGNISTREAM — DATA QUALITY SUMMARY")
print("=" * 55)

print("Rows:", quality_report["row_count"])
print("Columns:", quality_report["column_count"])
print("Duplicate rows:", quality_report["duplicate_rows"])

display(
    pd.DataFrame({
        "Missing_Count": raw_df.isna().sum(),
        "Missing_Percentage": (
            raw_df.isna().mean() * 100
        ).round(2),
    })
)

display(pd.DataFrame(quality_report["numeric_checks"]).T)

COGNISTREAM — DATA QUALITY SUMMARY
Rows: 500
Columns: 9
Duplicate rows: 0


,Missing_Count,Missing_Percentage
hours_coding,0,0.0
coffee_intake_mg,0,0.0
distractions,0,0.0
sleep_hours,0,0.0
commits,0,0.0
bugs_reported,0,0.0
ai_usage_hours,0,0.0
cognitive_load,0,0.0
task_success,0,0.0


,invalid_numeric_values,invalid_range_values
hours_coding,0,0
coffee_intake_mg,0,0
distractions,0,0
sleep_hours,0,0
commits,0,0
bugs_reported,0,0
ai_usage_hours,0,0
cognitive_load,0,0
task_success,0,0


In [7]:
display(raw_df.describe().T)

,count,mean,std,min,25%,50%,75%,max
hours_coding,500.0,5.01560,1.947253,0.0,3.6000,5.03,6.275,12.00
coffee_intake_mg,500.0,463.19200,142.334991,6.0,369.5000,500.50,600.000,600.00
distractions,500.0,2.97600,1.678409,0.0,2.0000,3.00,4.000,8.00
sleep_hours,500.0,6.97580,1.455965,3.0,6.1000,6.95,7.900,10.00
commits,500.0,4.60800,2.704990,0.0,3.0000,5.00,6.000,13.00
bugs_reported,500.0,0.85800,1.102846,0.0,0.0000,0.00,2.000,5.00
ai_usage_hours,500.0,1.51092,1.085557,0.0,0.6975,1.26,2.070,6.36
cognitive_load,500.0,4.49820,1.868861,1.0,3.1750,4.40,5.800,10.00
task_success,500.0,0.60600,0.489124,0.0,0.0000,1.00,1.000,1.00


In [8]:
print("Cognitive load distribution:")
display(
    raw_df["cognitive_load"]
    .value_counts()
    .sort_index()
    .rename_axis("Cognitive_Load")
    .to_frame("Count")
)

print("Task success distribution:")
display(
    raw_df["task_success"]
    .value_counts()
    .sort_index()
    .rename_axis("Task_Success")
    .to_frame("Count")
)

print(
    "Task success proportions:"
)
display(
    raw_df["task_success"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
    .rename("Percentage")
    .to_frame()
)

Cognitive load distribution:


,Count
Cognitive_Load,
1.0,16
1.1,1
1.2,2
1.3,3
1.4,4
...,...
8.8,2
9.1,3
9.5,1


Task success distribution:


,Count
Task_Success,
0,197
1,303


Task success proportions:


,Percentage
task_success,
0,39.4
1,60.6


In [9]:
import json

REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "data_collection_profile.json"
)

REPORT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with REPORT_PATH.open("w", encoding="utf-8") as file:
    json.dump(
        quality_report,
        file,
        indent=4,
        default=str,
    )

print("Report saved:", REPORT_PATH)

Report saved: C:\PlayGround\Infotact Data Analytics\CogniStream\reports\data_collection_profile.json


In [10]:
assert len(raw_df) == 500
assert len(raw_df.columns) == 9
assert schema_report["schema_valid"]
assert raw_df.isna().sum().sum() == 0
assert raw_df.duplicated().sum() == 0

print("Notebook 01 completed successfully.")
print("Raw dataset remains unchanged.")

Notebook 01 completed successfully.
Raw dataset remains unchanged.
